In [1]:
# Recharge automatiquement src/transformers.py quand il est modifié
%load_ext autoreload
%autoreload 2

# Rend le dossier src/ (un niveau au-dessus de notebooks/) importable
import sys
sys.path.append("..")
from src.transformers import GroupMedianImputer, ModeImputer, Log1pTransformer, OrdinalMapper

import pandas as pd
import numpy as np
from src.preprocessing import clean, remove_outliers, add_features

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import KFold, cross_val_score, cross_val_predict
from sklearn.linear_model import Lasso
from lightgbm import LGBMRegressor
from sklearn.base import clone

In [2]:

train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")
test_ids = test["Id"]  

train.shape, test.shape

((1460, 81), (1459, 80))

In [3]:
train = remove_outliers(train)

y = np.log1p(train["SalePrice"])

# Features uniquement : on retire l'identifiant et la cible
X_train = train.drop(columns=["Id", "SalePrice"])
X_test  = test.drop(columns=["Id"])

# Concaténation train + test ; n_train sert à re-séparer plus tard
n_train = len(X_train)
all_X = pd.concat([X_train, X_test], axis=0, ignore_index=True)
all_X.shape 

(2917, 79)

In [4]:
cleaned = clean(all_X)
cleaned_featured = add_features(cleaned)

In [5]:
X_train = cleaned_featured.iloc[:n_train].copy()
X_test = cleaned_featured.iloc[n_train:].copy()

In [6]:
print(X_train.shape) 
print(X_test.shape) 
print(y.shape) 

(1458, 80)
(1459, 80)
(1458,)


In [7]:
na = X_train.isna().sum()
print(na[na > 0])

LotFrontage     259
MasVnrType        5
BsmtExposure      1
BsmtFinType2      1
Electrical        1
dtype: int64


In [8]:
# Colonnes passées en log1p (section 4 + décisions de la section 6)
cols_log = [
    # Surfaces à longue traîne
    "LotArea", "LotFrontage", "GrLivArea", "1stFlrSF", "TotalBsmtSF", "BsmtUnfSF",
    # Beaucoup de zéros mais vraie quantité
    "2ndFlrSF", "WoodDeckSF", "OpenPorchSF", "MasVnrArea", "BsmtFinSF1",
    # Presque vides : log1p plutôt que TotalPorchSF (rejetée)
    "MiscVal", "EnclosedPorch", "ScreenPorch", "3SsnPorch",
    # Feature engineering
    "TotalSF",
]

In [9]:
# Échelle de qualité commune (None = absent)
QUAL = {"None": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5}

# Colonnes qui utilisent l'échelle de qualité Po → Ex
cols_qual = [
    "ExterQual", "ExterCond",
    "BsmtQual", "BsmtCond",
    "HeatingQC", "KitchenQual", "FireplaceQu",
    "GarageQual", "GarageCond",
]

mapping_qual = {col: QUAL for col in cols_qual}

# Échelle partagée par les deux types de finition du sous-sol
FINTYPE = {"None": 0, "Unf": 1, "LwQ": 2, "Rec": 3, "BLQ": 4, "ALQ": 5, "GLQ": 6}

# Ordinales avec leur propre échelle (du pire au meilleur)
mapping_other = {
    "BsmtExposure": {"None": 0, "No": 1, "Mn": 2, "Av": 3, "Gd": 4},
    "BsmtFinType1": FINTYPE,
    "BsmtFinType2": FINTYPE,
    "GarageFinish": {"None": 0, "Unf": 1, "RFn": 2, "Fin": 3},
    "Functional":   {"Sev": 0, "Maj2": 1, "Maj1": 2, "Mod": 3, "Min2": 4, "Min1": 5, "Typ": 6},
    "PavedDrive":   {"N": 0, "P": 1, "Y": 2},
    "CentralAir":   {"N": 0, "Y": 1},
    "Street":       {"Grvl": 0, "Pave": 1},
}

# Les 17 ordinales : qualité + échelles propres
mapping_ord = mapping_other | mapping_qual

In [10]:
# Nominales = colonnes texte qui ne sont pas ordinales (→ one-hot)
cols_cate = X_train.select_dtypes(exclude="number").columns.tolist()
cols_nom = [c for c in cols_cate if c not in mapping_ord]

In [11]:
# Les étapes du Pipeline (non entraînées) : 17 imputers, log1p, encodage ordinal, one-hot, standardisation ; fit sur le train uniquement
imp_lotfrontage    = GroupMedianImputer()                                         # médiane par Neighborhood
mode_masvnrtype    = ModeImputer()                                                # mode hors "None"

imp_garage_area    = GroupMedianImputer(target_col="GarageArea", group_col="GarageType")
imp_garage_cars    = GroupMedianImputer(target_col="GarageCars", group_col="GarageType")
mode_garage_finish = ModeImputer(target_col="GarageFinish")
mode_garage_qual   = ModeImputer(target_col="GarageQual")
mode_garage_cond   = ModeImputer(target_col="GarageCond")

mode_bsmt_cond     = ModeImputer(target_col="BsmtCond")
mode_bsmt_qual     = ModeImputer(target_col="BsmtQual")
mode_bsmt_exposure = ModeImputer(target_col="BsmtExposure")
mode_bsmt_fintype2 = ModeImputer(target_col="BsmtFinType2", exclude_value=["None", "Unf"])  # type aménagé le plus fréquent

mode_mszoning      = ModeImputer(target_col="MSZoning")
mode_exterior1st   = ModeImputer(target_col="Exterior1st")
mode_exterior2nd   = ModeImputer(target_col="Exterior2nd")
mode_electrical    = ModeImputer(target_col="Electrical")
mode_kitchenqual   = ModeImputer(target_col="KitchenQual")
mode_saletype      = ModeImputer(target_col="SaleType")

log_transfo        = Log1pTransformer(cols=cols_log)
ordinal_mapper     = OrdinalMapper(mapping=mapping_ord)
encodeur           = ColumnTransformer(
    transformers=[
        # One-hot des 25 nominales ; catégorie inconnue du test → 0 partout
        ("ohe", OneHotEncoder(handle_unknown="ignore", drop=None, sparse_output=False), cols_nom),
    ],
    remainder="passthrough",              # les 55 autres colonnes passent telles quelles
    verbose_feature_names_out=False,      # noms sans préfixe : GrLivArea, Neighborhood_NAmes
).set_output(transform="pandas")
scaleur            = StandardScaler().set_output(transform="pandas")   # moyenne 0, écart-type 1 (appris sur le train) : pénalité du Lasso équitable

In [12]:
# Pipeline (21 étapes : imputations, log1p, encodage ordinal, one-hot, standardisation) : fit sur le train uniquement, puis appliqué au train et au test
pipe = Pipeline([
    ("lotfrontage",    imp_lotfrontage),
    ("masvnrtype",     mode_masvnrtype),
    ("garage_area",    imp_garage_area),
    ("garage_cars",    imp_garage_cars),
    ("garage_finish",  mode_garage_finish),
    ("garage_qual",    mode_garage_qual),
    ("garage_cond",    mode_garage_cond),
    ("bsmt_cond",      mode_bsmt_cond),
    ("bsmt_qual",      mode_bsmt_qual),
    ("bsmt_exposure",  mode_bsmt_exposure),
    ("bsmt_fintype2",  mode_bsmt_fintype2),
    ("mszoning",       mode_mszoning),
    ("exterior1st",    mode_exterior1st),
    ("exterior2nd",    mode_exterior2nd),
    ("electrical",     mode_electrical),
    ("kitchenqual",    mode_kitchenqual),
    ("saletype",       mode_saletype),
    ("log_of_cols",    log_transfo),
    ("ordinal_map",    ordinal_mapper),
    ("one_hot_encod",  encodeur),
    ("standard_scal",  scaleur)
])

pipe.fit(X_train)
X_train_prep = pipe.transform(X_train)
X_test_prep  = pipe.transform(X_test)

In [13]:
# Contrôle de sortie du Pipeline
na_train = X_train_prep.isna().sum().sum()
na_test  = X_test_prep.isna().sum().sum()
print("NaN train / test               :", na_train, "/", na_test)                       # attendu : 0 / 0

print("Formes train / test            :", X_train_prep.shape, X_test_prep.shape)        # attendu : (1458, n) et (1459, n)
print("Mêmes colonnes, même ordre     :", list(X_train_prep.columns) == list(X_test_prep.columns))  # attendu : True
print("Colonnes créées par le one-hot :", X_train_prep.shape[1] - (80 - len(cols_nom)))

print("Type train / test              :", type(X_train_prep).__name__, "/", type(X_test_prep).__name__)  # attendu : DataFrame

cols_texte_restantes = X_train_prep.select_dtypes(exclude="number").columns.tolist()
print("Colonnes texte restantes       :", cols_texte_restantes)                         # attendu : []

# Log1p : vérifié AVANT le scaler (pipe[:-1] = toutes les étapes sauf la dernière)
X_train_avant_scaler = pipe[:-1].transform(X_train)
print("Max des colonnes en log1p, avant scaler (attendu < 13) :", X_train_avant_scaler[cols_log].max().max().round(2))

# Standardisation : sur le train, chaque colonne doit avoir moyenne 0 et écart-type 1
print("Train : |moyenne| max          :", X_train_prep.mean().abs().max().round(6))             # attendu : ≈ 0
print("Train : écart-type min / max   :", X_train_prep.std(ddof=0).min().round(4), "/",
      X_train_prep.std(ddof=0).max().round(4))                                                # attendu : 1 / 1

# Sur le test : proche de 0 et 1, mais pas exactement (centré avec la moyenne du train)
cols_exemple = ["GrLivArea", "OverallQual", "CentralAir"]
print(pd.DataFrame({
    "moy train": X_train_prep[cols_exemple].mean(),
    "moy test":  X_test_prep[cols_exemple].mean(),
    "std train": X_train_prep[cols_exemple].std(ddof=0),
    "std test":  X_test_prep[cols_exemple].std(ddof=0),
}).round(3))

NaN train / test               : 0 / 0
Formes train / test            : (1458, 253) (1459, 253)
Mêmes colonnes, même ordre     : True
Colonnes créées par le one-hot : 198
Type train / test              : DataFrame / DataFrame
Colonnes texte restantes       : []
Max des colonnes en log1p, avant scaler (attendu < 13) : 12.28
Train : |moyenne| max          : 0.0
Train : écart-type min / max   : 1.0 / 1.0
             moy train  moy test  std train  std test
GrLivArea         -0.0    -0.037        1.0     0.957
OverallQual       -0.0    -0.011        1.0     1.044
CentralAir         0.0    -0.016        1.0     1.028


In [14]:
# Réglage d'alpha : même CV pour chaque valeur, puis nombre de colonnes gardées
alphas = [0.002, 0.0025, 0.003, 0.004, 0.005, 0.006]
# Modèle complet : préparation (21 étapes) + Lasso
modele = Pipeline([
    ("prep", clone(pipe)),
    ("lasso", Lasso(alpha=0.0005, max_iter=50000)),
])

# CV 5 folds sur le train brut, mêmes parts pour tous les alphas (la préparation est réapprise à chaque fold)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

resultats = []
for a in alphas:
    modele.set_params(lasso__alpha=a)

    # Performance : CV sur des copies de modele
    scores = cross_val_score(modele, X_train, y, cv=kf, scoring="neg_root_mean_squared_error")
    rmse = -scores   # sklearn renvoie l'opposé de l'erreur (convention : plus grand = meilleur)

    # Fit sur tout le train : uniquement pour compter les coefficients non nuls
    modele.fit(X_train, y)
    n_colonnes = (modele.named_steps["lasso"].coef_ != 0).sum()

    resultats.append({
        "alpha": a,
        "rmse_folds": rmse.round(4),
        "rmse_moyenne": round(rmse.mean(), 4),
        "ecart_type": round(rmse.std(), 4),
        "n_colonnes": n_colonnes,
    })

df_resultats = pd.DataFrame(resultats)
df_resultats

,alpha,rmse_folds,rmse_moyenne,ecart_type,n_colonnes
0,0.0020,"[0.1119, 0.1244, 0.1231, 0.1194, 0.1015]",0.1161,0.0085,130
1,0.0025,"[0.1114, 0.1219, 0.1237, 0.1181, 0.101]",0.1152,0.0083,112
2,0.0030,"[0.1112, 0.1201, 0.1245, 0.1172, 0.1007]",0.1148,0.0082,102
3,0.0040,"[0.1114, 0.1175, 0.1256, 0.1161, 0.1008]",0.1143,0.0082,88
4,0.0050,"[0.1118, 0.1157, 0.1262, 0.1158, 0.1014]",0.1141,0.0080,66
5,0.0060,"[0.1124, 0.1144, 0.1266, 0.116, 0.102]",0.1143,0.0078,61


In [15]:
# 1. Meilleur alpha = plus petite RMSE moyenne en CV
idx_meilleur = df_resultats["rmse_moyenne"].idxmin()
meilleur_alpha = df_resultats.loc[idx_meilleur, "alpha"]
rmse_lasso = df_resultats.loc[df_resultats["alpha"] == meilleur_alpha, "rmse_folds"].iloc[0]   # ses 5 RMSE, pour comparer aux autres modèles

# 2. Après la boucle, modele garde le dernier alpha (0.01) : on remet le meilleur, PUIS on réentraîne
modele.set_params(lasso__alpha=meilleur_alpha)
modele.fit(X_train, y)

# 3. Coefficients avec le nom de leur colonne (feature_names_in_ : noms reçus par le Lasso, dans l'ordre de coef_)
coefs = pd.Series(modele.named_steps["lasso"].coef_,
                  index=modele.named_steps["lasso"].feature_names_in_)

print("Meilleur alpha    :", meilleur_alpha)
print("Colonnes gardées  :", (coefs != 0).sum(), "/", len(coefs))

Meilleur alpha    : 0.005
Colonnes gardées  : 66 / 253


In [16]:
coefs.sort_values()                         # du plus négatif au plus positif

HouseAge                -0.040133
MSZoning_C (all)        -0.025419
SaleCondition_Abnorml   -0.012466
RemodAge                -0.012380
KitchenAbvGr            -0.011842
                           ...   
OverallCond              0.035174
LotArea                  0.040156
TotalSF                  0.061290
GrLivArea                0.076035
OverallQual              0.077320
Length: 253, dtype: float64

In [17]:
coefs.abs().sort_values(ascending=False).head(15)   # les 15 plus fortes, quel que soit le signe

OverallQual             0.077320
GrLivArea               0.076035
TotalSF                 0.061290
LotArea                 0.040156
HouseAge                0.040133
OverallCond             0.035174
MSZoning_C (all)        0.025419
TotalBath               0.025300
GarageArea              0.021855
BsmtFinSF1              0.020516
Functional              0.019633
KitchenQual             0.018377
Neighborhood_Crawfor    0.017695
GarageCars              0.017626
1stFlrSF                0.014375
dtype: float64

In [18]:
coefs[["TotalSF", "IsRemodeled", "HasPool", "IsNew", "Has2ndFloor", "TotalBath","1stFlrSF","2ndFlrSF"]]

TotalSF        0.061290
IsRemodeled   -0.000000
HasPool        0.002466
IsNew          0.000000
Has2ndFloor   -0.000000
TotalBath      0.025300
1stFlrSF       0.014375
2ndFlrSF       0.000000
dtype: float64

In [19]:
# Soumission 1 : Lasso, alpha = 0.005 (RMSE CV 0.1141)
# 1. Prédiction du test : modele applique les 21 transform appris sur le train, puis le Lasso
pred_log = modele.predict(X_test)   # en log(1 + prix), comme y

# 2. Retour en dollars : expm1 est l'inverse exact de log1p
pred_prix = np.expm1(pred_log)

In [20]:
# 3. Fichier au format attendu par Kaggle (Id, SalePrice)
sample = pd.read_csv("../data/sample_submission.csv")
submission = pd.DataFrame({"Id": test_ids, "SalePrice": pred_prix})
submission.head()

,Id,SalePrice
0,1461,120166.594422
1,1462,157270.876451
2,1463,177523.268773
3,1464,195112.364271
4,1465,204288.405817


In [21]:
# 4. Vérifications avant envoi
print("Lignes                      :", len(submission), "/ attendu", len(sample))
print("Mêmes colonnes que l'exemple:", list(submission.columns) == list(sample.columns))
print("Mêmes Id, même ordre        :", (submission["Id"].values == sample["Id"].values).all())
print("Id min / max                :", submission["Id"].min(), "/", submission["Id"].max())   # attendu : 1461 / 2919
print("NaN / prix <= 0             :", submission["SalePrice"].isna().sum(), "/", (submission["SalePrice"] <= 0).sum())

# Prix plausibles : même ordre de grandeur que les prix du train
pd.DataFrame({
    "prédictions test": submission["SalePrice"].describe(),
    "prix du train":    train["SalePrice"].describe(),
}).round(0)

Lignes                      : 1459 / attendu 1459
Mêmes colonnes que l'exemple: True
Mêmes Id, même ordre        : True
Id min / max                : 1461 / 2919
NaN / prix <= 0             : 0 / 0


,prédictions test,prix du train
count,1459.0,1458.0
mean,177703.0,180933.0
std,72959.0,79495.0
min,41144.0,34900.0
25%,127513.0,129925.0
50%,158776.0,163000.0
75%,211783.0,214000.0
max,705875.0,755000.0


In [22]:
# 5. Enregistrement : le nom dit quel modèle a produit le fichier
submission.to_csv("../submissions/lasso_alpha0005.csv", index=False)   # index=False : pas de colonne d'index en trop

In [23]:
# LightGBM : même préparation (clone : indépendante de celle du Lasso), même CV (kf)
modele_lgbm = Pipeline([
    ("prep", clone(pipe)),
    ("lgbm", LGBMRegressor(
        n_estimators=1000,       # nombre d'arbres
        learning_rate=0.03,      # part de chaque correction ajoutée
        num_leaves=15,           # petits arbres : 15 feuilles maximum
        min_child_samples=10,    # au moins 10 maisons par feuille
        subsample=0.8,           # chaque arbre voit 80 % des maisons...
        subsample_freq=1,        # ...retirées au hasard à chaque arbre
        colsample_bytree=0.5,    # chaque arbre voit 50 % des colonnes
        random_state=42,         # tirages reproductibles
        verbose=-1,              # pas de messages pendant l'entraînement
    )),
])

scores = cross_val_score(modele_lgbm, X_train, y, cv=kf, scoring="neg_root_mean_squared_error")
rmse_lgbm = -scores

# Comparaison fold par fold avec le meilleur Lasso (mêmes parts) : différence < 0 = LightGBM meilleur
comparaison = pd.DataFrame({
    "lasso": rmse_lasso,
    "lgbm": rmse_lgbm.round(4),
    "différence": (rmse_lgbm - rmse_lasso).round(4),
}, index=[f"fold {i}" for i in range(1, 6)])
comparaison.loc["moyenne"] = comparaison.mean().round(4)
comparaison.loc["écart-type"] = [round(rmse_lasso.std(), 4), round(rmse_lgbm.std(), 4), None]
comparaison

,lasso,lgbm,différence
fold 1,0.1118,0.1220,0.0102
fold 2,0.1157,0.1094,-0.0063
fold 3,0.1262,0.1228,-0.0034
fold 4,0.1158,0.1316,0.0158
fold 5,0.1014,0.1060,0.0046
moyenne,0.1142,0.1184,0.0042
écart-type,0.0080,0.0094,NaN


In [24]:
# Prédictions out-of-fold : chaque maison est prédite par un modèle qui ne l'a pas vue (même kf)
oof_lasso = cross_val_predict(modele, X_train, y, cv=kf)        # modele : Lasso, alpha = 0.005
oof_lgbm  = cross_val_predict(modele_lgbm, X_train, y, cv=kf)

# RMSE sur les 1 458 prédictions (proche, mais pas égale, de la moyenne des 5 RMSE par fold)
rmse_lasso_oof = np.sqrt(np.mean((y - oof_lasso) ** 2))
rmse_lgbm_oof  = np.sqrt(np.mean((y - oof_lgbm) ** 2))

# ρ : à quel point les deux modèles se trompent pareil (1 = mêmes erreurs, combinaison inutile)
rho = np.corrcoef(y - oof_lasso, y - oof_lgbm)[0, 1]

print(f"RMSE out-of-fold Lasso    : {rmse_lasso_oof:.4f}")
print(f"RMSE out-of-fold LightGBM : {rmse_lgbm_oof:.4f}")
print(f"Corrélation des erreurs ρ : {rho:.3f}   (combinaison utile si < {rmse_lasso_oof / rmse_lgbm_oof:.3f})")

RMSE out-of-fold Lasso    : 0.1144
RMSE out-of-fold LightGBM : 0.1187
Corrélation des erreurs ρ : 0.819   (combinaison utile si < 0.964)


In [25]:
# Combinaison Lasso + LightGBM : poids w du Lasso, de 0 (LightGBM seul) à 1 (Lasso seul)
# Pas de réentraînement : on combine les prédictions out-of-fold déjà calculées
poids = np.linspace(0, 1, 11)   # 11 points réguliers, sans erreur d'arrondi (contrairement à arange)

resultats_w = []
for w in poids:
    oof_comb = w * oof_lasso + (1 - w) * oof_lgbm          # prédiction combinée des 1 458 maisons
    rmse_comb = np.sqrt(np.mean((y - oof_comb) ** 2))
    resultats_w.append({
        "w": round(w, 1),   # arrondi : permet les comparaisons w == 0 et w == 1
        "rmse": rmse_comb,
    })

df_w = pd.DataFrame(resultats_w)
df_w

,w,rmse
0,0.0,0.118736
1,0.1,0.116416
2,0.2,0.114482
3,0.3,0.112953
4,0.4,0.111846
5,0.5,0.111174
6,0.6,0.110944
7,0.7,0.111160
8,0.8,0.111818
9,0.9,0.112911


In [26]:
# Meilleur poids, et contrôles : w = 1 doit redonner le Lasso seul (0.1144), w = 0 LightGBM seul (0.1187)
idx_meilleur_w = df_w["rmse"].idxmin()
meilleur_w = df_w.loc[idx_meilleur_w, "w"]
meilleur_rmse = df_w.loc[idx_meilleur_w, "rmse"]

rmse_w_1 = df_w.loc[df_w["w"] == 1, "rmse"].iloc[0]
rmse_w_0 = df_w.loc[df_w["w"] == 0, "rmse"].iloc[0]

print(f"Meilleur w = {meilleur_w:.1f} -> RMSE = {meilleur_rmse:.4f}")
print(f"w = 1 (Lasso seul)    -> RMSE = {rmse_w_1:.4f}")
print(f"w = 0 (LightGBM seul) -> RMSE = {rmse_w_0:.4f}")
print(f"Gain sur le Lasso seul : {meilleur_rmse - rmse_w_1:.4f}")

Meilleur w = 0.6 -> RMSE = 0.1109
w = 1 (Lasso seul)    -> RMSE = 0.1144
w = 0 (LightGBM seul) -> RMSE = 0.1187
Gain sur le Lasso seul : -0.0035


In [27]:
# Soumission 2 : combinaison 60 % Lasso + 40 % LightGBM (RMSE out-of-fold 0.1109)
# 1. Chaque modèle est entraîné sur tout le train (la CV n'entraînait que des clones)
modele.fit(X_train, y)
modele_lgbm.fit(X_train, y)

# 2. Chaque modèle prédit le test, en log(1 + prix)
pred_log_lasso = modele.predict(X_test)
pred_log_lgbm  = modele_lgbm.predict(X_test)

# 3. On combine les PRÉDICTIONS (pas les modèles), en log, avec le poids choisi en CV
pred_log_modele_comb = meilleur_w * pred_log_lasso + (1 - meilleur_w) * pred_log_lgbm

# 4. Retour en dollars
pred_prix_modele_comb = np.expm1(pred_log_modele_comb)

In [28]:
# Fichier au format attendu par Kaggle (Id, SalePrice)
submission_modele_comb = pd.DataFrame({"Id": test_ids, "SalePrice": pred_prix_modele_comb})
submission_modele_comb.head()

,Id,SalePrice
0,1461,120946.585780
1,1462,158183.007248
2,1463,179222.135566
3,1464,194293.370971
4,1465,198930.618358


In [29]:
# Vérifications avant envoi
print("Lignes                      :", len(submission_modele_comb), "/ attendu", len(sample))
print("Mêmes colonnes que l'exemple:", list(submission_modele_comb.columns) == list(sample.columns))
print("Mêmes Id, même ordre        :", (submission_modele_comb["Id"].values == sample["Id"].values).all())
print("Id min / max                :", submission_modele_comb["Id"].min(), "/", submission_modele_comb["Id"].max())   # attendu : 1461 / 2919
print("NaN / prix <= 0             :", submission_modele_comb["SalePrice"].isna().sum(), "/", (submission_modele_comb["SalePrice"] <= 0).sum())

# Maison 2550 (jumelle des anomalies retirées du train) : LightGBM devrait faire baisser sa prédiction
id_2550 = submission_modele_comb["Id"] == 2550
print("Maison 2550, Lasso seul     :", round(submission.loc[id_2550, "SalePrice"].iloc[0]))
print("Maison 2550, combinaison    :", round(submission_modele_comb.loc[id_2550, "SalePrice"].iloc[0]))

# Prix plausibles : combinaison, Lasso seul (soumission 1) et prix du train
pd.DataFrame({
    "combinaison":   submission_modele_comb["SalePrice"].describe(),
    "Lasso seul":    submission["SalePrice"].describe(),
    "prix du train": train["SalePrice"].describe(),
}).round(0)

Lignes                      : 1459 / attendu 1459
Mêmes colonnes que l'exemple: True
Mêmes Id, même ordre        : True
Id min / max                : 1461 / 2919
NaN / prix <= 0             : 0 / 0
Maison 2550, Lasso seul     : 705875
Maison 2550, combinaison    : 698543


,combinaison,Lasso seul,prix du train
count,1459.0,1459.0,1458.0
mean,177977.0,177703.0,180933.0
std,75321.0,72959.0,79495.0
min,43575.0,41144.0,34900.0
25%,127254.0,127513.0,129925.0
50%,157165.0,158776.0,163000.0
75%,210043.0,211783.0,214000.0
max,698543.0,705875.0,755000.0


In [30]:
# Enregistrement : le nom dit quels modèles et quels poids ont produit le fichier
submission_modele_comb.to_csv("../submissions/blend_lasso06_lgbm04.csv", index=False)